# Maintenance Work Order Data Workflow

This notebook develops a reproducible Python data workflow for a synthetic CMMS-style maintenance dataset. The records are simulated for educational purposes and do not represent real customer, technician, property, or asset data.

The current project phase focuses on data generation, ingestion, cleaning, exploratory analysis, visualization, and interpretation as a foundation for later machine-learning work-order priority prediction.


## 1. Setup

Import the Python libraries used in the workflow and make the repository root available on the Python import path.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import RAW_DATA_FILE, PROCESSED_DATA_FILE
from src.generate_data import save_dataset
from src.cleaning import (
    standardize_categories,
    remove_duplicate_work_orders,
    flag_invalid_values,
    impute_missing_values,
)

pd.set_option('display.max_columns', None)
print(f'Project root: {project_root}')
print(f'Raw data path: {RAW_DATA_FILE}')
print(f'Processed data path: {PROCESSED_DATA_FILE}')


## 2. Data Ingestion

The raw CSV is generated reproducibly when it is not already present. The dataset is then loaded with Pandas, and the first rows are displayed as required by the project rubric.

The parser uses `keep_default_na=False` with `na_values=['']`. This is important because **`None` is a valid value for `occupancy_impact`**, but Pandas can otherwise interpret the literal word `None` as a missing value. Empty CSV fields are still treated as missing data.


In [ ]:
if not RAW_DATA_FILE.exists():
    generated_df = save_dataset()
    print(f'Generated {len(generated_df):,} synthetic raw rows.')
else:
    print('Using the existing synthetic raw CSV.')

df_raw = pd.read_csv(
    RAW_DATA_FILE,
    parse_dates=['created_date'],
    keep_default_na=False,
    na_values=[''],
)
print(f'Loaded {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns.')


In [ ]:
df_raw.head()


### Initial Structure Check

Before cleaning, inspect dimensions, data types, missing values, duplicate work-order IDs, and category counts. This establishes what problems the cleaning functions need to address.


In [ ]:
print('Shape:', df_raw.shape)
print('\nData types:')
display(df_raw.dtypes.to_frame('dtype'))

print('Missing values:')
display(
    df_raw.isna().sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_frame('missing_count')
)

duplicate_ids = df_raw.duplicated(subset='work_order_id').sum()
print(f'Duplicate work-order IDs: {duplicate_ids}')
print(f'Raw asset-type labels: {df_raw["asset_type"].nunique()}')
print(f'Raw priority labels: {df_raw["priority"].nunique()}')
print(f'Rows whose occupancy impact is the valid category "None": {(df_raw["occupancy_impact"] == "None").sum()}')


## 3. Data Cleaning

The raw dataset intentionally contains missing values, inconsistent capitalization and whitespace, duplicate work orders, and a small number of impossible or injected out-of-range values. Four reusable cleaning functions in `src/cleaning.py` are applied below. Every function includes an informative docstring.

### Cleaning decisions and justification

1. **Standardize categories.** Equivalent labels such as `HVAC`, `hvac`, and ` HVAC ` must represent one category. Without normalization, grouping and later ML encoding would incorrectly treat them as different values.
2. **Remove duplicate work orders.** Duplicate work-order IDs would double-count maintenance events and bias frequency, cost, and priority summaries.
3. **Flag invalid values.** Negative asset ages are impossible. Resolution times above 240 hours and repair costs above $25,000 are outside the documented valid range of this synthetic generator and were intentionally injected as quality problems. They are changed to missing values before imputation.
4. **Impute missing values.** Missing asset condition is labeled `Unknown` instead of inventing a Good/Fair/Poor condition. Missing numerical values are filled using the median for the same asset type and maintenance type, with an overall median fallback. This preserves rows while limiting sensitivity to skewed cost and duration distributions.

These thresholds and imputation rules are appropriate for this **synthetic educational dataset**. They must be re-evaluated rather than copied blindly when real client data becomes available.


In [ ]:
raw_quality = pd.Series({
    'rows': len(df_raw),
    'duplicate_work_order_ids': df_raw.duplicated(subset='work_order_id').sum(),
    'missing_values': df_raw.isna().sum().sum(),
    'unique_asset_type_labels': df_raw['asset_type'].nunique(),
    'unique_priority_labels': df_raw['priority'].nunique(),
    'negative_asset_ages': (df_raw['asset_age_years'] < 0).sum(),
    'resolution_hours_over_240': (df_raw['resolution_hours'] > 240).sum(),
    'repair_cost_over_25000': (df_raw['estimated_repair_cost'] > 25000).sum(),
}, name='raw')

raw_quality.to_frame()


In [ ]:
df_clean = standardize_categories(df_raw)
df_clean = remove_duplicate_work_orders(df_clean)
df_clean = flag_invalid_values(df_clean)
df_clean = impute_missing_values(df_clean)

clean_quality = pd.Series({
    'rows': len(df_clean),
    'duplicate_work_order_ids': df_clean.duplicated(subset='work_order_id').sum(),
    'missing_values': df_clean.isna().sum().sum(),
    'unique_asset_type_labels': df_clean['asset_type'].nunique(),
    'unique_priority_labels': df_clean['priority'].nunique(),
    'negative_asset_ages': (df_clean['asset_age_years'] < 0).sum(),
    'resolution_hours_over_240': (df_clean['resolution_hours'] > 240).sum(),
    'repair_cost_over_25000': (df_clean['estimated_repair_cost'] > 25000).sum(),
}, name='clean')

cleaning_comparison = pd.concat([raw_quality, clean_quality], axis=1)
cleaning_comparison


### Cleaning result and bias considerations

The comparison above demonstrates the effect of the cleaning functions rather than hiding the changes. Duplicate rows are removed, categorical labels collapse to their canonical groups, impossible synthetic values are handled, and missing values are resolved.

Cleaning can itself introduce bias. For example, deleting every row with a missing field could disproportionately remove certain asset types or priority classes if documentation quality differs by group. This workflow therefore preserves rows where possible. Even median imputation is not neutral: it reduces variability and can pull unusual records toward a typical group value. For that reason, the original raw data is kept unchanged, the cleaning rules are explicit, and the project documents that these rules must be validated on real operational data before production use.


In [ ]:
PROCESSED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_DATA_FILE, index=False)
print(f'Saved {len(df_clean):,} cleaned rows to {PROCESSED_DATA_FILE}')
df_clean.head()


## 4. Exploratory Data Analysis

Reusable EDA functions, summary statistics, grouping, and filtering will be implemented here to explore maintenance patterns and future ML features.


## 5. Visualizations

At least three Matplotlib visualizations with descriptive titles, labeled axes, and written interpretations will be implemented here.


## 6. Summary and Interpretation

The final section will summarize insights, patterns, assumptions, limitations, synthetic-data caveats, and implications for future machine-learning work.
